In [10]:
from transformers import MarianMTModel , MarianTokenizer

model_name= "Helsinki-NLP/opus-mt-es-en"

tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)

text = "Este libro trsta  sobre  ingenieria electrica."
inputs = tokenizer(text, return_tensors="pt", padding=True)

translated = model.generate(**inputs)

print(tokenizer.decode(translated[0], skip_special_tokens=True))

This book about electrical engineering.


In [11]:
!pip install pdfplumber

In [4]:
import pdfplumber

text = ""
with pdfplumber.open("Crea y divaga Invent and Wander.pdf") as pdf:
  for page in pdf.pages:
    text += page.extract_text() + "/n"




In [5]:
import re

def clean_text(text):
  text = re.sub(r'-\n', '', text)

  # For fixing broken words
  text = re.sub(r'\n+', '\n', text)
  return text.strip()

## This step alone improves translation quality a lot


In [6]:
import nltk

nltk.download("punkt_tab")
from nltk.tokenize import sent_tokenize

sentence = sent_tokenize(clean_text(text))


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [7]:
def chunk_sentences(sentences, max_sentences=5):
  for i in range(0, len(sentences), max_sentences):
    yield "".join(sentences[i:i+max_sentences])

## Batch Translation it is efficient and stable


In [8]:
from transformers import MarianMTModel, MarianTokenizer

import torch

model_name= "Helsinki-NLP/opus-mt-es-en"

tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)

def translate_batch(texts):
  inputs = tokenizer(texts, return_tensors="pt", padding=True)
  with torch.no_grad():
    outputs = model.generate(**inputs, max_length=512)
    return [tokenizer.decode(o, skip_special_tokens=True) for o in outputs]

In [9]:
# Assuming 'sentence' is the list from your sent_tokenize step
translated_ebook = []
batch_size = 8

for i in range(0, len(sentence), batch_size):
    batch = sentence[i:i + batch_size]
    translated_batch = translate_batch(batch)
    translated_ebook.extend(translated_batch)

    if i % 100 == 0:
        print(f"Progress: {i}/{len(sentence)} sentences translated.")

# Save your final text
final_text = " ".join(translated_ebook)


Progress: 0/4226 sentences translated.
Progress: 200/4226 sentences translated.
Progress: 400/4226 sentences translated.
Progress: 600/4226 sentences translated.
Progress: 800/4226 sentences translated.
Progress: 1000/4226 sentences translated.
Progress: 1200/4226 sentences translated.
Progress: 1400/4226 sentences translated.
Progress: 1600/4226 sentences translated.
Progress: 1800/4226 sentences translated.
Progress: 2000/4226 sentences translated.
Progress: 2200/4226 sentences translated.
Progress: 2400/4226 sentences translated.
Progress: 2600/4226 sentences translated.
Progress: 2800/4226 sentences translated.
Progress: 3000/4226 sentences translated.
Progress: 3200/4226 sentences translated.
Progress: 3400/4226 sentences translated.
Progress: 3600/4226 sentences translated.
Progress: 3800/4226 sentences translated.
Progress: 4000/4226 sentences translated.
Progress: 4200/4226 sentences translated.


In [13]:
with open("Translated_Ebook_EN.txt", "w", encoding="utf-8") as f:
  f.write(final_text)

print("File saved successfully...!")

File saved successfully...!


In [14]:
from google.colab import  files
files.download("Translated_Ebook_EN.txt")
#

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>